## Instalando bibliotecas

In [1]:
!pip install transformers datasets peft accelerate torch

## Importando bibliotecas

In [2]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
import torch

/home/jessica/Área de trabalho/topicosAvancadosEmIAAAvaliacao02/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#!pip install --upgrade torchao

## Preparando Dataset

In [3]:
def convert_to_hf_format(example):
    """Combina instrução e saída em um único texto."""
    return {
        "text": f"Instruction: {example['Instruction']}\nOutput: {example['Output']}"
    }

# Carrega o arquivo JSON Lines
dataset = load_dataset('json', data_files='../dataset_sem_curadoria.jsonl')
# Aplica a conversão
dataset = dataset.map(convert_to_hf_format)
# Divide em treino e teste
dataset = dataset["train"].train_test_split(test_size=0.2)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['Instruction', 'Output', 'text'],
        num_rows: 89
    })
    test: Dataset({
        features: ['Instruction', 'Output', 'text'],
        num_rows: 23
    })
})


## Carregar o Modelos Pré-Treinado e o Tokenizador

In [4]:
model_name = "TucanoBR/Tucano-1b1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Modelo carregado: {model_name}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3741.72it/s]


Modelo carregado: TucanoBR/Tucano-1b1


## Testando Modelo Crú

In [7]:
def generate_response(model, tokenizer, instruction, input_text=""):
    """Gera uma resposta a partir de uma instrução, usando o modelo fornecido."""
    if input_text:
        prompt = f"Instruction: {instruction}\nInput: {input_text}\nOutput:"
    else:
        prompt = f"Instruction: {instruction}\nOutput:"

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=150,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,          # ativa amostragem para usar temperatura
        temperature=0.7
    )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extrai apenas a parte após "Output:"
    resposta = full_output.split("Output:")[-1].strip()
    return resposta

# Exemplo de instrução (deve existir no dataset.jsonl)
test_instruction = "Qual é a prioridade ao inspecionar o local de trabalho antes de usar a máquina?"

print("=== ANTES DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta base: {generate_response(base_model, tokenizer, test_instruction)}")

=== ANTES DO FINE-TUNING ===
Instrução: Qual é a prioridade ao inspecionar o local de trabalho antes de usar a máquina?


Resposta base: Se você notar uma área que precisa ser limpa, certifique-se de seguir os passos para verificar as áreas problemáticas e tome medidas corretivas antes de qualquer coisa.
Priority1: Prepare seu espaço de trabalho antes de começar;
Preparing Your Work Space Before Starting: 
We are going to switch some lights and furnishes for the workplace. You have ready your work space by way of a rack attached both inside and out.
Priority1: Preparar seu espaço de trabalho ;
Preparing Your Workspace Before Beginning: 
If you've got any questions, please let me know so that I


## Tokenização do Dataset

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)
print("Dataset tokenizado:", tokenized_datasets)

## Preparar o Modelo para LoRA

In [ ]:
# A partir daqui, usaremos uma cópia do modelo base para o fine-tuning
model = base_model  # poderia também ser uma nova instância
model = prepare_model_for_kbit_training(model)

## Verificando Camadas Target

In [ ]:
import torch.nn as nn

for name, module in base_model.named_modules():
    if isinstance(module, nn.Linear):
        print(name)

## Configurar e Injetar LoRA

In [ ]:
lora_config = LoraConfig(
    r=16,                    # rank da decomposição
    lora_alpha=32,           # escala alpha
    target_modules=["q_proj", "k_proj"],  # camadas alvo , "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    inference_mode=False     # False = modo treinamento
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Data Collator (Modelagem Causal)

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

## Argumentos de Treinamento (Preparando Hiperparâmetros)

In [ ]:
training_args = TrainingArguments(
    output_dir="./modelos_treinados",          # diretório de saída
    eval_strategy="steps",
    eval_steps=100,
    learning_rate=1e-3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    report_to="none",               # desabilita logging para wandb/mlflow
)

## Iniciar Treino

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
)

## Treinar Modelo

In [ ]:
trainer.train()

## Salvar o Modelo Ajustado e o Tokenizador

In [ ]:
model.save_pretrained("lora_tucano_causal_finetuned_model")
tokenizer.save_pretrained("tucano_tokenizer")

## Inferência

In [ ]:
# Carrega o modelo fine-tunado (apenas os adaptadores LoRA)
finetuned_model = AutoModelForCausalLM.from_pretrained("lora_tucano_causal_finetuned_model")
finetuned_tokenizer = AutoTokenizer.from_pretrained("tucano_tokenizer")

# Garante que o token de padding esteja definido
if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

In [ ]:
print("=== DEPOIS DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta ajustada: {generate_response(finetuned_model, finetuned_tokenizer, test_instruction)}")